# SocrataProfiler

Perfilador ligero de datasets Socrata para el proyecto RAIZ. Permite comparar fuentes antes de descargarlas y ayuda a decidir cuáles tienen cobertura temporal, estructura y granularidad útiles.

Este notebook **no descarga datasets completos** y no ejecuta agregaciones globales como `count(*)` o `distinct`, que ya han producido timeouts en fuentes grandes.

## 1. Configuración

### Uso rápido

1. Pegue en `DATASETS` URLs de API como `https://www.datos.gov.co/resource/xxxx-xxxx.json` o IDs como `xxxx-xxxx`.
2. Si hace falta, indique la columna de fecha en `DATE_COLUMN_OVERRIDES`.
3. Active `EJECUTAR_PERFILADO = True`.
4. Ejecute el notebook de principio a fin.

También se admite una entrada con nombre descriptivo: `{'url': '...', 'nombre': 'Temperatura'}`.

In [ ]:
import re
from datetime import datetime, timezone
from urllib.parse import urlparse

import pandas as pd
import requests

try:
    from IPython.display import Markdown, display
except ImportError:
    Markdown = str

    def display(valor):
        print(valor)

# Acepta URLs, IDs o diccionarios con las llaves 'url'/'id' y 'nombre'.
DATASETS = [
    # {'url': 'https://www.datos.gov.co/resource/uejq-wxrr.json', 'nombre': 'EVA 2019-2024'},
    # 'https://www.datos.gov.co/resource/s54a-sgyg.json',
]

# Banderita de seguridad: con False, Run all no consulta ninguna API.
EJECUTAR_PERFILADO = False

SAMPLE_SIZE = 10
CADENCE_SAMPLE_SIZE = 100
MAX_TEMPORAL_COLUMNS = 3
REQUEST_TIMEOUT = 45
APP_TOKEN = None  # Opcional. No escribir tokens reales antes de hacer commit.

# Opcionales, usando el ID como llave.
# Ejemplo: {'xxxx-xxxx': 'fechaobservacion'}
DATE_COLUMN_OVERRIDES = {}

# Filtros SoQL pequeños para perfilar una entidad o territorio cuando sea necesario.
# Ejemplo: {'xxxx-xxxx': "departamento = 'BOYACÁ'"}
WHERE_BY_DATASET = {}

print({
    'datasets_configurados': len(DATASETS),
    'ejecutar_perfilado': EJECUTAR_PERFILADO,
    'muestra_filas': SAMPLE_SIZE,
    'muestra_temporal': CADENCE_SAMPLE_SIZE,
})

## 2. Funciones de acceso y metadatos

El esquema se obtiene del endpoint de metadatos de Socrata, por lo que `tipo_socrata` es más confiable que inferir tipos desde diez filas JSON. Las consultas de datos usan `LIMIT` dentro de `$query`, siguiendo el comportamiento observado en datos.gov.co.

In [ ]:
DATASET_ID_PATTERN = re.compile(r'([a-z0-9]{4}-[a-z0-9]{4})', re.IGNORECASE)
VALID_FIELD_PATTERN = re.compile(r'^[A-Za-z_][A-Za-z0-9_]*$')


def interpretar_referencia_dataset(referencia):
    """Normaliza una URL, un ID o un diccionario de configuración."""
    if isinstance(referencia, dict):
        valor = referencia.get('url') or referencia.get('id')
        nombre = referencia.get('nombre')
    else:
        valor = referencia
        nombre = None

    if not valor:
        raise ValueError(f'Referencia sin URL o ID: {referencia!r}')

    valor = str(valor).strip()
    coincidencia = DATASET_ID_PATTERN.search(valor)
    if not coincidencia:
        raise ValueError(f'No se encontró un ID Socrata xxxx-xxxx en: {valor}')

    dataset_id = coincidencia.group(1).lower()
    parsed = urlparse(valor if '://' in valor else '')
    dominio = parsed.netloc or 'www.datos.gov.co'
    return {
        'dataset_id': dataset_id,
        'dominio': dominio,
        'nombre_configurado': nombre,
        'referencia_original': valor,
        'api_url': f'https://{dominio}/resource/{dataset_id}.json',
        'metadata_url': f'https://{dominio}/api/views/{dataset_id}',
    }


def headers_socrata():
    headers = {'Accept': 'application/json', 'User-Agent': 'RAIZ-SocrataProfiler/1.0'}
    if APP_TOKEN:
        headers['X-App-Token'] = APP_TOKEN
    return headers


def consultar_json(url, params=None, timeout=REQUEST_TIMEOUT):
    response = requests.get(
        url,
        params=params,
        headers=headers_socrata(),
        timeout=timeout,
    )
    response.raise_for_status()
    return response.json()


def consultar_soql(config, select='*', where=None, order=None, limit=10):
    partes = [f'SELECT {select}']
    if where:
        partes.append(f'WHERE {where}')
    if order:
        partes.append(f'ORDER BY {order}')
    partes.append(f'LIMIT {int(limit)}')
    datos = consultar_json(config['api_url'], params={'$query': ' '.join(partes)})
    return pd.DataFrame(datos)


def citar_campo(nombre_campo):
    if not VALID_FIELD_PATTERN.fullmatch(str(nombre_campo)):
        raise ValueError(f'Nombre de campo no seguro para SoQL: {nombre_campo!r}')
    return f'`{nombre_campo}`'


def combinar_where(where_base, condicion):
    if where_base:
        return f'({where_base}) AND ({condicion})'
    return condicion


def epoch_a_iso(valor):
    if valor in (None, ''):
        return None
    try:
        return datetime.fromtimestamp(int(valor), tz=timezone.utc).isoformat()
    except (TypeError, ValueError, OSError):
        return str(valor)

In [ ]:
def columnas_desde_metadata(metadata, dataset_id):
    filas = []
    for posicion, columna in enumerate(metadata.get('columns', []), start=1):
        cached = columna.get('cachedContents') or {}
        filas.append({
            'dataset_id': dataset_id,
            'posicion': posicion,
            'columna': columna.get('name'),
            'campo_api': columna.get('fieldName'),
            'tipo_socrata': columna.get('dataTypeName'),
            'descripcion': columna.get('description'),
            'nulos_metadata': cached.get('null'),
            'no_nulos_metadata': cached.get('non_null'),
            'min_metadata': cached.get('smallest'),
            'max_metadata': cached.get('largest'),
        })
    return pd.DataFrame(filas)


def columnas_temporales(catalogo_columnas, dataset_id):
    override = DATE_COLUMN_OVERRIDES.get(dataset_id)
    if override:
        return [override] if isinstance(override, str) else list(override)

    if catalogo_columnas.empty:
        return []

    tipos_fecha = {'calendar_date', 'floating_timestamp', 'fixed_timestamp'}
    patron_nombre = re.compile(
        r'fecha|date|timestamp|datetime|(^|_)hora($|_)|año|(^|_)(ano|anio|year)($|_)',
        re.IGNORECASE,
    )
    candidatas = []
    for _, fila in catalogo_columnas.iterrows():
        campo = fila.get('campo_api')
        nombre_visible = fila.get('columna')
        tipo = str(fila.get('tipo_socrata') or '').lower()
        texto_columna = f'{campo or ""} {nombre_visible or ""}'
        if campo and (tipo in tipos_fecha or patron_nombre.search(texto_columna)):
            candidatas.append(campo)

    return list(dict.fromkeys(candidatas))[:MAX_TEMPORAL_COLUMNS]


def inferir_tipo_pandas_muestra(muestra):
    if muestra.empty:
        return {}
    return {columna: str(muestra[columna].dtype) for columna in muestra.columns}

## 3. Perfil temporal

Para cada columna temporal candidata se consultan una fila inicial, una final y una ventana reciente de marcas de tiempo. La granularidad resultante es **aparente**, no definitiva. En datasets con muchas estaciones o entidades, mezclar sus relojes puede hacer que el conjunto parezca más frecuente de lo que reporta cada entidad individual. Use `WHERE_BY_DATASET` para filtrar una estación o municipio cuando necesite validar la frecuencia real.

In [ ]:
def convertir_valores_temporales(serie, campo):
    numerica = pd.to_numeric(serie, errors='coerce')

    if numerica.notna().mean() >= 0.8:
        anios_validos = numerica.between(1800, 2200)
        if anios_validos.mean() >= 0.8:
            return pd.to_datetime(
                numerica.round().astype('Int64').astype(str),
                format='%Y',
                errors='coerce',
            )

    return pd.to_datetime(serie, errors='coerce')


def clasificar_granularidad(delta_mediana_segundos):
    if pd.isna(delta_mediana_segundos):
        return 'no_determinada'

    horas = delta_mediana_segundos / 3600
    dias = horas / 24
    if horas < 18:
        return 'subdiaria'
    if dias <= 2:
        return 'diaria'
    if 5 <= dias <= 10:
        return 'semanal'
    if 25 <= dias <= 35:
        return 'mensual'
    if 75 <= dias <= 105:
        return 'trimestral'
    if 150 <= dias <= 215:
        return 'semestral'
    if 330 <= dias <= 400:
        return 'anual'
    return 'irregular_o_no_clasificada'


def consultar_extremo_temporal(config, campo, direccion, where_base):
    campo_soql = citar_campo(campo)
    where = combinar_where(where_base, f'{campo_soql} IS NOT NULL')
    resultado = consultar_soql(
        config,
        select=campo_soql,
        where=where,
        order=f'{campo_soql} {direccion}',
        limit=1,
    )
    if resultado.empty or campo not in resultado.columns:
        return None
    return resultado.iloc[0][campo]


def perfilar_columna_temporal(config, campo, catalogo_columnas, where_base=None):
    resultado = {
        'dataset_id': config['dataset_id'],
        'campo_temporal': campo,
        'fecha_min': None,
        'fecha_max': None,
        'granularidad_aparente': 'no_determinada',
        'confianza_granularidad': 'no_determinada',
        'marcas_unicas_muestra': 0,
        'delta_mediana_minutos': None,
        'delta_p10_minutos': None,
        'delta_p90_minutos': None,
        'advertencia': None,
    }

    fila_metadata = catalogo_columnas[catalogo_columnas['campo_api'] == campo]
    es_campo_anio = False
    if not fila_metadata.empty:
        resultado['fecha_min'] = fila_metadata.iloc[0].get('min_metadata')
        resultado['fecha_max'] = fila_metadata.iloc[0].get('max_metadata')
        nombre_visible = str(fila_metadata.iloc[0].get('columna') or '')
        es_campo_anio = bool(re.search(r'año|(^|_)(ano|anio|year)($|_)', nombre_visible, re.IGNORECASE))

    errores = []
    if es_campo_anio:
        resultado['granularidad_aparente'] = 'anual'
        resultado['confianza_granularidad'] = 'alta'
        try:
            campo_soql = citar_campo(campo)
            rango = consultar_soql(
                config,
                select=f'min({campo_soql}) as valor_min, max({campo_soql}) as valor_max',
                where=where_base,
                limit=1,
            )
            if not rango.empty:
                resultado['fecha_min'] = rango.iloc[0].get('valor_min') or resultado['fecha_min']
                resultado['fecha_max'] = rango.iloc[0].get('valor_max') or resultado['fecha_max']
        except Exception as exc:
            errores.append(f'rango anual: {type(exc).__name__}: {exc}')
        if errores:
            resultado['advertencia'] = ' | '.join(errores)
        return resultado

    try:
        resultado['fecha_min'] = consultar_extremo_temporal(
            config, campo, 'ASC', where_base
        ) or resultado['fecha_min']
    except Exception as exc:
        errores.append(f'mínimo: {type(exc).__name__}: {exc}')

    try:
        resultado['fecha_max'] = consultar_extremo_temporal(
            config, campo, 'DESC', where_base
        ) or resultado['fecha_max']
    except Exception as exc:
        errores.append(f'máximo: {type(exc).__name__}: {exc}')

    try:
        campo_soql = citar_campo(campo)
        where = combinar_where(where_base, f'{campo_soql} IS NOT NULL')
        muestra = consultar_soql(
            config,
            select=campo_soql,
            where=where,
            order=f'{campo_soql} DESC',
            limit=CADENCE_SAMPLE_SIZE,
        )
        if campo in muestra.columns:
            fechas = convertir_valores_temporales(muestra[campo], campo).dropna()
            fechas = pd.Series(fechas.unique()).sort_values().reset_index(drop=True)
            resultado['marcas_unicas_muestra'] = len(fechas)
            if es_campo_anio:
                resultado['granularidad_aparente'] = 'anual'
                resultado['confianza_granularidad'] = 'alta'
            elif len(fechas) >= 20:
                resultado['confianza_granularidad'] = 'alta'
            elif len(fechas) >= 5:
                resultado['confianza_granularidad'] = 'media'
            elif len(fechas) >= 2:
                resultado['confianza_granularidad'] = 'baja'
                errores.append(
                    f'granularidad: solo {len(fechas)} marcas temporales únicas en la muestra'
                )
            deltas = fechas.diff().dropna().dt.total_seconds()
            deltas = deltas[deltas > 0]
            if not deltas.empty:
                mediana = deltas.median()
                resultado['delta_mediana_minutos'] = round(mediana / 60, 2)
                resultado['delta_p10_minutos'] = round(deltas.quantile(0.10) / 60, 2)
                resultado['delta_p90_minutos'] = round(deltas.quantile(0.90) / 60, 2)
                if not es_campo_anio:
                    resultado['granularidad_aparente'] = clasificar_granularidad(mediana)
    except Exception as exc:
        errores.append(f'granularidad: {type(exc).__name__}: {exc}')

    if errores:
        resultado['advertencia'] = ' | '.join(errores)
    return resultado

## 4. Ejecución del perfilado

Cada dataset se procesa de forma independiente. Si una consulta falla o excede el tiempo, se conserva el esquema y la muestra que sí hayan podido recuperarse, y el error queda registrado como advertencia.

In [ ]:
def perfilar_dataset(referencia):
    config = interpretar_referencia_dataset(referencia)
    dataset_id = config['dataset_id']
    errores = []

    try:
        metadata = consultar_json(config['metadata_url'])
    except Exception as exc:
        metadata = {}
        errores.append(f'metadatos: {type(exc).__name__}: {exc}')

    columnas = columnas_desde_metadata(metadata, dataset_id)
    where_base = WHERE_BY_DATASET.get(dataset_id)

    try:
        muestra = consultar_soql(
            config,
            where=where_base,
            limit=SAMPLE_SIZE,
        )
    except Exception as exc:
        muestra = pd.DataFrame()
        errores.append(f'muestra: {type(exc).__name__}: {exc}')

    tipos_muestra = inferir_tipo_pandas_muestra(muestra)
    if not columnas.empty:
        columnas['tipo_pandas_muestra'] = columnas['campo_api'].map(tipos_muestra)

    temporales = []
    for campo in columnas_temporales(columnas, dataset_id):
        temporales.append(
            perfilar_columna_temporal(
                config,
                campo,
                columnas,
                where_base=where_base,
            )
        )

    resumen = {
        'dataset_id': dataset_id,
        'nombre': config['nombre_configurado'] or metadata.get('name'),
        'dominio': config['dominio'],
        'api_url': config['api_url'],
        'categoria': metadata.get('category'),
        'actualizado_utc': epoch_a_iso(metadata.get('rowsUpdatedAt')),
        'columnas': len(columnas),
        'filas_muestra': len(muestra),
        'columnas_temporales': ', '.join(columnas_temporales(columnas, dataset_id)),
        'filtro_aplicado': where_base,
        'estado': 'ok' if not errores else 'parcial',
        'advertencias': ' | '.join(errores) or None,
    }

    return {
        'config': config,
        'metadata': metadata,
        'resumen': resumen,
        'columnas': columnas,
        'muestra': muestra,
        'temporal': pd.DataFrame(temporales),
    }

In [ ]:
perfiles = []

if not EJECUTAR_PERFILADO:
    print('Perfilado desactivado. Pegue las URLs en DATASETS y cambie EJECUTAR_PERFILADO a True.')
elif not DATASETS:
    print('No hay datasets configurados en DATASETS.')
else:
    for indice, referencia in enumerate(DATASETS, start=1):
        print(f'[{indice}/{len(DATASETS)}] Perfilando {referencia!r}...')
        try:
            perfil = perfilar_dataset(referencia)
            perfiles.append(perfil)
            resumen = perfil['resumen']
            display(Markdown(f"### {resumen['dataset_id']} — {resumen['nombre'] or 'Sin título'}"))
            display(pd.DataFrame([resumen]))
            display(Markdown('**Esquema**'))
            display(perfil['columnas'])
            display(Markdown(f"**Muestra de hasta {SAMPLE_SIZE} filas**"))
            display(perfil['muestra'])
            display(Markdown('**Perfil temporal**'))
            display(perfil['temporal'])
        except Exception as exc:
            print(f'No fue posible perfilar esta referencia: {type(exc).__name__}: {exc}')

## 5. Catálogos consolidados

Estos DataFrames reúnen los resultados de la corrida y quedan listos para comparar o exportar cuando se haya decidido cuáles fuentes son viables.

In [ ]:
catalogo_datasets = pd.DataFrame([perfil['resumen'] for perfil in perfiles])
catalogo_columnas = (
    pd.concat([perfil['columnas'] for perfil in perfiles], ignore_index=True)
    if perfiles
    else pd.DataFrame()
)
catalogo_temporal = (
    pd.concat(
        [perfil['temporal'] for perfil in perfiles if not perfil['temporal'].empty],
        ignore_index=True,
    )
    if any(not perfil['temporal'].empty for perfil in perfiles)
    else pd.DataFrame()
)
muestras = {perfil['resumen']['dataset_id']: perfil['muestra'] for perfil in perfiles}

display(Markdown('### Resumen de datasets'))
display(catalogo_datasets)
display(Markdown('### Cobertura y granularidad temporal'))
display(catalogo_temporal)

## 6. Cómo interpretar el resultado

- `tipo_socrata` describe el tipo declarado por la fuente; `tipo_pandas_muestra` muestra cómo llegó en JSON.
- `fecha_min` y `fecha_max` se intentan consultar en vivo. Si la consulta falla, pueden quedar los extremos cacheados en metadatos.
- `granularidad_aparente` se estima sobre las marcas temporales más recientes y no prueba que todo el histórico mantenga esa frecuencia.
- `confianza_granularidad` baja cuando la muestra contiene muy pocas marcas temporales únicas, algo común cuando muchas estaciones reportan simultáneamente.
- En datasets por estación, municipio u otra entidad, valide la frecuencia con un filtro en `WHERE_BY_DATASET`.
- Un timeout no significa automáticamente que el dataset sea inútil: significa que esa consulta no es viable como diagnóstico rápido.

La salida de este notebook sirve para catalogar y priorizar fuentes. La auditoría profunda debe hacerse después sobre el subconjunto descargado o sobre una fuente ya agregada.